<a href="https://colab.research.google.com/github/VijayR-Nair/embedded-pump-predictive-maintenance/blob/main/smartsensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Core imports and reproducibility
import os
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import joblib

import tensorflow as tf
from tensorflow.keras import layers, Model

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_PATH = Path('/content/sensor.csv')
ARTIFACT_DIR = Path('/content/anomaly_edge_artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


print('TensorFlow version:', tf.__version__)
print('Artifacts will be saved to:', ARTIFACT_DIR)

In [ ]:
# Load the dataset
sensor = pd.read_csv(DATA_PATH)

print(sensor.shape)
print(list(sensor.columns)[:10], '...')
print('\nMachine status counts:')
print(sensor['machine_status'].value_counts(dropna=False))

In [ ]:
all_sensors = [c for c in sensor.columns if c.startswith('sensor_')]
sensor_cols = [c for c in all_sensors if not sensor[c].isna().all()]
empty_sensor_cols = sorted(list(set(all_sensors) - set(sensor_cols)))

print(empty_sensor_cols)

In [ ]:
normal_sensor = sensor[sensor['machine_status'] == 'NORMAL'].copy()

X_normal = normal_sensor[sensor_cols]
X_all = sensor[sensor_cols]


X_train_raw, X_val_raw = train_test_split(
    X_normal,
    test_size=0.20,
    shuffle=False    #to keep the time order
)

# print(X_train_raw.shape)
# print(X_val_raw.shape)

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_train_raw)      #replaces 'NaN' values with median of the column
X_val_imputed = imputer.transform(X_val_raw)
X_all_imputed = imputer.transform(X_all)

X_train = scaler.fit_transform(X_train_imputed).astype('float32')
X_val = scaler.transform(X_val_imputed).astype('float32')
X_scaled = scaler.transform(X_all_imputed).astype('float32')

# print(X_train.shape)
# print(X_val.shape)
# print(X_scaled.shape)

input_dim = X_train.shape[1]

inputs = layers.Input(shape=(input_dim,), name='sensor_input')
x = layers.Dense(16, activation='relu', name='encoder_dense_16')(inputs)
bottleneck = layers.Dense(8, activation='relu', name='bottleneck')(x)
x = layers.Dense(16, activation='relu', name='decoder_dense_16')(bottleneck)
outputs = layers.Dense(input_dim, activation='linear', name='reconstruction')(x)

autoencoder = Model(inputs, outputs, name='dense_sensor_autoencoder')
autoencoder.compile(optimizer='adam', loss= 'mse')
autoencoder.summary()


In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(                          #stops training when there's not any change in validation loss, waits for 8 more epochs before
    monitor='val_loss',                                                 #stopping (patience) and True option  enables to take the best weight
    patience=8,
    restore_best_weights=True
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-5,
    verbose=1
)


#At the start of training, a larger learning rate helps the model learn faster.
#Later, when improvement slows down, a smaller learning rate can help the model
#make finer adjustments instead of jumping around the best solution.

history = autoencoder.fit(
    X_train,
    X_train,                                                                              #an autoencoder learns to reconstruct its own input. So the input and the target output are the same
    validation_data=(X_val, X_val),
    epochs=120,
    batch_size=75,
    callbacks=[early_stop, reduce_lr],
    shuffle=True,
    verbose=1
)

In [ ]:
# Plot training history
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('MSE reconstruction loss')
plt.title('Autoencoder training history')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
THRESHOLD_PERCENTILE = 99.0

train_recon = autoencoder.predict(X_train, verbose=0)
val_recon = autoencoder.predict(X_val, verbose=0)
all_recon = autoencoder.predict(X_scaled, verbose=0)

train_error = np.mean(np.square(X_train - train_recon), axis=1)
val_error = np.mean(np.square(X_val - val_recon), axis=1)
reconstruction_error = np.mean(np.square(X_scaled - all_recon), axis=1)

threshold = float(np.percentile(val_error, THRESHOLD_PERCENTILE))                                #Threshold percentile filters the anomalies more efficiently (99.0)

sensor['reconstruction_error'] = reconstruction_error
sensor['is_anomaly'] = sensor['reconstruction_error'] > threshold


print(f'Anomaly threshold: {threshold:.8f}')
print('\nPredicted anomaly counts:')
print(sensor['is_anomaly'].value_counts())


In [ ]:
y_true = (sensor['machine_status'] != 'NORMAL').astype(int)
y_pred = sensor['is_anomaly'].astype(int)

print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y_true, y_pred))

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'ANOMALY'], zero_division=0))

if y_true.nunique() == 2:
    print('ROC AUC:', roc_auc_score(y_true, sensor['reconstruction_error']))
    print('Average precision:', average_precision_score(y_true, sensor['reconstruction_error']))

print('\nAverage reconstruction error by machine_status:')
print(sensor.groupby('machine_status')['reconstruction_error'].agg(['count', 'mean', 'median', 'max']).sort_values('mean', ascending=False))


In [ ]:
# Visualize reconstruction error over the dataset
plt.figure(figsize=(14, 5))
plt.plot(sensor.index, sensor['reconstruction_error'], linewidth=0.8, label='Reconstruction error')
plt.axhline(threshold, linestyle='--', label='Threshold')
plt.xlabel('Row index / time order')
plt.ylabel('Reconstruction error')
plt.title('Anomaly score over time')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
sensor.boxplot(column='reconstruction_error', by='machine_status', rot=45)
plt.title('Reconstruction error by machine status')
plt.suptitle('')
plt.ylabel('Reconstruction error')
plt.show()


In [ ]:
# Save model and preprocessing artifacts
keras_model_path = ARTIFACT_DIR / 'autoencoder.keras'
imputer_path = ARTIFACT_DIR / 'imputer.pkl'
scaler_path = ARTIFACT_DIR / 'scaler.pkl'
metadata_path = ARTIFACT_DIR / 'metadata.json'
predictions_path = ARTIFACT_DIR / 'sensor_predictions.csv'


autoencoder.save(keras_model_path)
joblib.dump(imputer, imputer_path)
joblib.dump(scaler, scaler_path)

metadata = {
    'sensor_cols': sensor_cols,
    'empty_sensor_cols': empty_sensor_cols,
    'threshold': threshold,
    'threshold_percentile': THRESHOLD_PERCENTILE,
    'label_rule': 'machine_status != NORMAL is treated as anomaly for evaluation',
    'input_dim': int(input_dim),
    'model_name': autoencoder.name,
    'imputer_strategy': 'median',
    'scaler': 'StandardScaler',
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

sensor.to_csv(predictions_path, index=False)

print('Saved:')
for path in [keras_model_path, imputer_path, scaler_path, metadata_path, predictions_path]:
    print('-', path)


In [ ]:

model_path = 'autoencoder.keras'

model = tf.keras.models.load_model(model_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)

#Convert and save the model
tflite_model = converter.convert()

with open('autoencoder.tflite', 'wb') as f:
    f.write(tflite_model)

print("Successfully converted to autoencoder.tflite!")

In [ ]:
def representative_dataset():
  for i in range(100):
    sample = X_train[i:i+1].astype(np.float32)
    # The model has a single input tensor, get its name from model.inputs[0].name
    yield {model.inputs[0].name: sample}

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset


converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

with open("autoencoder_full_int8.tflite", "wb") as f:
    f.write(tflite_int8_model)

print("Saved full int8 TFLite model.")